# 07 — Live FAA airport status with Spark Structured Streaming

The historical pipeline (notebooks 02–06) explains *why* delays cascade. This notebook adds the **live** half: the FAA NAS Status feed (public XML, updates about every minute) is polled, each poll is saved as a raw file (**bronze**), Spark Structured Streaming parses new files into events (**silver**), and the events are joined to the cascade-prone airports from the historical analysis (**gold** `live_status` in Atlas, read by the dashboard).

Runs locally (the Dataproc cluster has no internet access). `MODE="live"` polls the FAA; `MODE="replay"` re-uses the saved sample file in `samples/` for offline testing.

In [2]:
import os, re, json, time, threading, shutil, glob
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

MODE = "live"          # "live" = poll the FAA feed, "replay" = copy the saved sample file
POLL_SECONDS = 60      # the feed updates about once a minute; do not poll faster than ~30 s
N_POLLS = 5            # number of polls in this run
FEED_URL = "https://nasstatus.faa.gov/api/airport-status-information"

ROOT = Path.cwd()
RAW  = ROOT / "data/live/raw"       # bronze: one raw XML per poll
EVT  = ROOT / "data/live/events"    # silver: parsed events (Parquet)
CKPT = ROOT / "data/live/_ckpt"
SAMPLE = ROOT / "samples/airport_status_2026-11-20.xml"
STATS = ROOT / "data/propagation/tables/cascades_by_airport.csv"
for p in (RAW, EVT): p.mkdir(parents=True, exist_ok=True)
shutil.rmtree(CKPT, ignore_errors=True)
print(MODE, POLL_SECONDS, N_POLLS)

live 60 5


## 1. Poller (bronze)
Each poll is written to `data/live/raw/status_<UTC timestamp>.xml`, unchanged.

In [3]:
def poll_once(i):
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + f"{i:02d}Z"
    out = RAW / f"status_{ts}.xml"
    if MODE == "live":
        req = urllib.request.Request(FEED_URL, headers={"User-Agent": "UChicago ADSP32030 class project"})
        with urllib.request.urlopen(req, timeout=20) as r:
            out.write_bytes(r.read())
    else:
        shutil.copy(SAMPLE, out)
    return out

def poller():
    for i in range(N_POLLS):
        try:
            f = poll_once(i); print("saved", f.name)
        except Exception as e:
            print("poll failed:", e)
        if i < N_POLLS - 1:
            time.sleep(POLL_SECONDS if MODE == "live" else 2)

## 2. Parser
The feed mixes time formats (update time in GMT, ground-stop end time in local time, closure start/reopen in UTC without a year), so raw strings are kept next to the parsed values. An airport that is **not** listed has no reported event. Closures include long-term NOTAMs "closed to non-scheduled transient GA", which do not affect airline flights, so they are flagged `ga_only` and excluded from the watchlist.

In [4]:
def mins(txt):
    """'1 hour and 37 minutes' -> 97, '25 minutes' -> 25, '1 hour' -> 60"""
    if not txt: return None
    h = re.search(r"(\d+)\s*hour", txt); m = re.search(r"(\d+)\s*minute", txt)
    if not (h or m): return None
    return (int(h.group(1)) * 60 if h else 0) + (int(m.group(1)) if m else 0)

def category(reason):
    r = (reason or "").upper()
    if "WX" in r or "WEATHER" in r or "VISIBILITY" in r or "THUNDER" in r or "FOG" in r or "WIND" in r or "SNOW" in r or "ICE" in r: return "weather"
    if "VOL" in r or "VOLUME" in r: return "volume"
    if "RWY" in r or "RUNWAY" in r: return "runway"
    if "NOTAM" in r or r.startswith("!"): return "notam"
    return "other"

def txt(el, tag):
    x = el.find(tag)
    return x.text.strip() if x is not None and x.text else None

def parse_status(xml_text, fetched_at, source_file):
    root = ET.fromstring(xml_text)
    upd = txt(root, "Update_Time")
    try: upd_utc = datetime.strptime(upd, "%a %b %d %H:%M:%S %Y GMT")
    except Exception: upd_utc = None
    base = dict(feed_update_utc=upd_utc, fetched_at_utc=fetched_at, source_file=source_file)
    rows = []
    for dt in root.findall("Delay_type"):
        name = txt(dt, "Name") or ""
        for p in dt.findall(".//Program"):
            rows.append(dict(base, airport=txt(p, "ARPT"), event_type="ground_stop", reason=txt(p, "Reason"),
                             category=category(txt(p, "Reason")), ends_local=txt(p, "End_Time")))
        for g in dt.findall(".//Ground_Delay"):
            rows.append(dict(base, airport=txt(g, "ARPT"), event_type="ground_delay", reason=txt(g, "Reason"),
                             category=category(txt(g, "Reason")), avg_min=mins(txt(g, "Avg")), max_min=mins(txt(g, "Max"))))
        for d in dt.findall(".//Delay"):
            ad = d.find("Arrival_Departure")
            rows.append(dict(base, airport=txt(d, "ARPT"), event_type="delay", reason=txt(d, "Reason"),
                             category=category(txt(d, "Reason")),
                             direction=ad.get("Type") if ad is not None else None,
                             min_min=mins(txt(ad, "Min")) if ad is not None else None,
                             max_min=mins(txt(ad, "Max")) if ad is not None else None,
                             trend=txt(ad, "Trend") if ad is not None else None))
        for a in dt.findall(".//Airport"):
            rs = txt(a, "Reason") or ""
            rows.append(dict(base, airport=txt(a, "ARPT"), event_type="closure", reason=rs, category="notam",
                             closure_start=txt(a, "Start"), reopen=txt(a, "Reopen"),
                             ga_only="NON SKED TRANSIENT GA" in rs.upper()))
    return rows

# test on the saved sample
_t = parse_status(SAMPLE.read_text(), datetime.utcnow(), "sample")
pd.DataFrame(_t)[["airport","event_type","category","direction","min_min","max_min","avg_min","trend","ga_only"]]

,airport,event_type,category,direction,min_min,max_min,avg_min,trend,ga_only
0,TEB,ground_stop,volume,NaN,NaN,NaN,NaN,NaN,NaN
1,SAN,ground_delay,volume,NaN,NaN,97.0,25.0,NaN,NaN
2,FLL,delay,weather,Departure,46.0,60.0,NaN,Increasing,NaN
3,SEA,delay,weather,Departure,16.0,30.0,NaN,Increasing,NaN
4,SFO,delay,runway,Departure,46.0,60.0,NaN,Increasing,NaN
5,TEB,delay,volume,Departure,31.0,45.0,NaN,Increasing,NaN
6,SNA,closure,notam,NaN,NaN,NaN,NaN,NaN,False
7,LAX,closure,notam,NaN,NaN,NaN,NaN,NaN,True
8,SAN,closure,notam,NaN,NaN,NaN,NaN,NaN,True
9,LAS,closure,notam,NaN,NaN,NaN,NaN,NaN,True


## 3. Airport context from the historical analysis
Cascade counts per airport (from notebook 04) tell us which live delays happen at airports that tend to start cascades.

In [5]:
stats = pd.read_csv(STATS)
print(stats.columns.tolist())
stats.head()

['airport', 'cascades', 'departures', 'per_1000']


,airport,cascades,departures,per_1000
0,JAX,9092,417600,21.772031
1,SAF,800,30771,25.998505
2,ANC,7362,295659,24.900307
3,MHK,425,22270,19.083969
4,COU,565,22052,25.621259


In [6]:
# normalise column names of the 04 table
cols = {c.lower(): c for c in stats.columns}
ap_col   = next(cols[c] for c in cols if c in ("airport","origin","iata","root_origin"))
casc_col = next(cols[c] for c in cols if "cascade" in c and "per" not in c)
rate_col = next((cols[c] for c in cols if "per_1000" in c or "per1000" in c or "rate" in c), None)
ctx = stats[[ap_col, casc_col] + ([rate_col] if rate_col else [])].copy()
ctx.columns = ["airport", "cascades"] + (["per_1000"] if rate_col else [])
ctx["cascade_rank"] = ctx["cascades"].rank(ascending=False, method="min").astype(int)
TOP_N = 15
ctx["cascade_prone"] = (ctx["cascade_rank"] <= TOP_N) | (ctx.get("per_1000", 0) >= 40)
ctx.sort_values("cascade_rank").head(10)

,airport,cascades,per_1000,cascade_rank,cascade_prone
213,DFW,272540,62.311265,1,True
386,ATL,262857,44.680570,2,True
32,ORD,260877,55.587079,3,True
128,DEN,193265,47.068866,4,True
74,CLT,129650,48.210252,5,True
102,PHX,107154,37.922847,6,True
342,IAH,105299,43.773812,7,True
350,LAX,104016,31.440864,8,True
78,LAS,97905,38.392160,9,True
112,MCO,97315,44.719501,10,True


## 4. Spark Structured Streaming
`readStream` watches `data/live/raw/`; each new XML file becomes one micro-batch (`maxFilesPerTrigger=1`). `foreachBatch` parses the file, joins the events to the airport context and appends them to the Parquet silver table.

In [7]:
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.master("local[2]").appName("faa_live")
         .config("spark.sql.shuffle.partitions", "4").config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

SCHEMA = T.StructType([
    T.StructField("airport", T.StringType()), T.StructField("event_type", T.StringType()),
    T.StructField("category", T.StringType()), T.StructField("reason", T.StringType()),
    T.StructField("direction", T.StringType()), T.StructField("min_min", T.IntegerType()),
    T.StructField("avg_min", T.IntegerType()), T.StructField("max_min", T.IntegerType()),
    T.StructField("trend", T.StringType()), T.StructField("ends_local", T.StringType()),
    T.StructField("closure_start", T.StringType()), T.StructField("reopen", T.StringType()),
    T.StructField("ga_only", T.BooleanType()), T.StructField("feed_update_utc", T.TimestampType()),
    T.StructField("fetched_at_utc", T.TimestampType()), T.StructField("source_file", T.StringType())])

ctx_sdf = F.broadcast(spark.createDataFrame(ctx))

def process_batch(batch_df, batch_id):
    rows = []
    for r in batch_df.collect():
        name = os.path.basename(r["source_file"])
        m = re.search(r"status_(\d{8}T\d{6})", name)
        fetched = datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")if m else datetime.utcnow()
        try: rows += parse_status(r["value"], fetched, name)
        except Exception as e: print("parse error", name, e)
    if not rows:
        return
    names = [f.name for f in SCHEMA.fields]
    ev = spark.createDataFrame([tuple(r.get(n) for n in names) for r in rows], SCHEMA)
    ev = (ev.join(ctx_sdf, "airport", "left")
            .withColumn("ga_only", F.coalesce("ga_only", F.lit(False)))
            .withColumn("watch", F.coalesce(F.col("cascade_prone"), F.lit(False)) & ~F.col("ga_only")).drop("cascade_prone"))
    ev.write.mode("append").parquet(str(EVT))
    print(f"batch {batch_id}: {ev.count()} events from {rows[0]['source_file']}")

stream = (spark.readStream.format("text").option("wholetext", "true").option("maxFilesPerTrigger", 1)
          .load(str(RAW)).withColumn("source_file", F.input_file_name()))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/20 15:22:58 WARN Utils: Your hostname, A-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.150.16.200 instead (on interface en0)
26/11/20 15:22:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 15:22:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 15:23:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


4.3.0


In [8]:
# clear previous run, then poll + stream together
for p in glob.glob(str(RAW / "*.xml")): os.remove(p)
shutil.rmtree(EVT, ignore_errors=True); EVT.mkdir(parents=True, exist_ok=True)

t = threading.Thread(target=poller); t.start()
q = (stream.writeStream.foreachBatch(process_batch).option("checkpointLocation", str(CKPT))
     .trigger(processingTime="5 seconds").start())

t.join()                     # all polls saved
time.sleep(8)                # let the last file be picked up
q.processAllAvailable()
q.stop()
print("stream stopped; files:", len(glob.glob(str(RAW / "*.xml"))))

saved status_20261120T21232400Z.xml
batch 0: 12 events from status_20261120T21232400Z.xml
saved status_20261120T21242401Z.xml
batch 1: 12 events from status_20261120T21242401Z.xml
saved status_20261120T21252402Z.xml
batch 2: 12 events from status_20261120T21252402Z.xml
saved status_20261120T21262403Z.xml
batch 3: 12 events from status_20261120T21262403Z.xml
saved status_20261120T21272504Z.xml
batch 4: 13 events from status_20261120T21272504Z.xml
stream stopped; files: 5


## 5. Results (silver)

In [9]:
ev = spark.read.parquet(str(EVT))
print("events:", ev.count(), "| polls:", ev.select("source_file").distinct().count())
ev.groupBy("event_type", "category").count().orderBy(F.desc("count")).show()

events: 61 | polls: 5
+------------+--------+-----+
|  event_type|category|count|
+------------+--------+-----+
|     closure|   notam|   20|
|       delay| weather|   16|
|       delay|  volume|   10|
|       delay|  runway|    5|
| ground_stop|  volume|    5|
|ground_delay|  volume|    5|
+------------+--------+-----+



In [10]:
# latest snapshot = events from the newest poll
latest = ev.filter(F.col("source_file") == ev.agg(F.max("source_file")).first()[0])
snap = latest.toPandas()
snap = snap.sort_values(["watch", "cascade_rank"], ascending=[False, True], na_position="last")
snap[["airport","event_type","category","reason","direction","min_min","max_min","avg_min","trend","ga_only","cascades","cascade_rank","watch"]]

,airport,event_type,category,reason,direction,min_min,max_min,avg_min,trend,ga_only,cascades,cascade_rank,watch
10,MIA,delay,weather,TM Initiatives:MIT:WX,Departure,16.0,30.0,NaN,Increasing,False,63243.0,20.0,True
9,FLL,delay,weather,TM Initiatives:MIT:WX,Departure,61.0,75.0,NaN,Increasing,False,62591.0,21.0,True
4,LAX,closure,notam,!LAX 05/277 LAX AD AP CLSD TO NON SKED TRANSIE...,None,NaN,NaN,NaN,None,True,104016.0,8.0,False
6,LAS,closure,notam,!LAS 10/160 LAS AD AP CLSD TO NON SKED TRANSIE...,None,NaN,NaN,NaN,None,True,97905.0,9.0,False
0,SFO,delay,runway,RWY:Rwy Change - Operational Necessity,Departure,61.0,75.0,NaN,Increasing,False,78938.0,16.0,False
12,SEA,delay,weather,WX:Low Visibility,Departure,16.0,30.0,NaN,Increasing,False,71769.0,19.0,False
5,SAN,closure,notam,!SAN 10/004 SAN AD AP CLSD TO NON SKED TRANSIE...,None,NaN,NaN,NaN,None,True,35997.0,29.0,False
8,SAN,ground_delay,volume,airport volume,None,NaN,97.0,25.0,None,False,35997.0,29.0,False
2,TPA,delay,weather,WX:Thunderstorms,Departure,61.0,75.0,NaN,Increasing,False,33861.0,31.0,False
3,SNA,closure,notam,!SNA 11/020 SNA AD AP CLSD DLY 0730-1415 26111...,None,NaN,NaN,NaN,None,False,17068.0,43.0,False


In [11]:
# Watchlist: live airline-relevant events at cascade-prone airports
watch = snap[(snap["watch"]) ].copy()
print("airports with active airline-relevant events:", snap[~snap.ga_only]["airport"].nunique())
print("of which cascade-prone:", watch["airport"].nunique())
watch[["airport","event_type","category","reason","min_min","max_min","avg_min","cascade_rank"]]

airports with active airline-relevant events: 9
of which cascade-prone: 2


,airport,event_type,category,reason,min_min,max_min,avg_min,cascade_rank
10,MIA,delay,weather,TM Initiatives:MIT:WX,16.0,30.0,NaN,20.0
9,FLL,delay,weather,TM Initiatives:MIT:WX,61.0,75.0,NaN,21.0


## 6. Gold: latest snapshot to Atlas
The snapshot replaces the `live_status` collection (small, ~dozens of documents). Credentials come from `.env` and are never printed. Without an Atlas URI the cell only reports a dry run.

In [12]:
def read_env(path=ROOT / ".env"):
    d = {}
    if path.exists():
        for line in path.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1); d[k.strip()] = v.strip().strip('"').strip("'")
    return d

env = read_env()
docs = json.loads(snap.assign(
    feed_update_utc=snap.feed_update_utc.astype(str), fetched_at_utc=snap.fetched_at_utc.astype(str)
).where(snap.notna(), None).to_json(orient="records"))
for d in docs: d["watch"] = bool(d.get("watch"))

if env.get("ATLAS_URI"):
    from pymongo import MongoClient
    cl = MongoClient(env["ATLAS_URI"], serverSelectionTimeoutMS=20000)
    col = cl[env.get("ATLAS_DB", "flightdelay")]["live_status"]
    col.delete_many({})
    if docs: col.insert_many(docs)
    col.create_index("airport")
    print("live_status documents:", col.count_documents({}))
    cl.close()
else:
    print("dry run, no ATLAS_URI; documents:", len(docs))

live_status documents: 13


In [13]:
spark.stop()